# Lösning: Labb 02 – Tsunami över havet: hastighet, uppgrundning och svamplager

Detta är en lösningsnotebook till `labs/02_tsunami_shoaling_sv.md`.

Målet är att visa tre saker:

1. En lång våg går snabbare på djupt vatten än på grunt vatten.
2. När vågen kommer in över grundare vatten kan våghöjden öka: detta kallas **uppgrundning**.
3. Ett **svamplager** vid den öppna randen dämpar oönskade reflektioner.

Modellen är fortfarande förenklad. Den visar inte vågbrytning, översvämning på land eller verklig risk för en specifik kust.


## 1. Importer och sökvägar

Koden nedan gör att notebooken fungerar både när den körs från repo-roten och inifrån `notebooks/`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

# Gör src/ importerbar även om notebooken körs från notebooks/.
HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / "src").exists() else HERE.parent
SRC_DIR = PROJECT_ROOT / "src"
ANIMATION_DIR = PROJECT_ROOT / "animations"
ANIMATION_DIR.mkdir(exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from shallowwater import (
    ModelParams,
    make_grid,
    zero_forcing,
    setup_initial_state,
    compute_dt_cfl,
    run_model,
    animate_eta,
    shelf_bathymetry,
    wave_speed,
    make_sponge_hook,
    sponge_mask_eta,
)


## 2. Skapa en bassäng med djup ocean och grund kust

Vi lägger kusten till höger, alltså i öster. Vattnet är djupt till vänster och blir gradvis grundare nära kusten.


In [ ]:
Nx, Ny = 160, 36
Lx, Ly = 2_400e3, 500e3

grid = make_grid(Nx, Ny, Lx, Ly)

H_deep = 4000.0       # djup ocean [m]
H_coast = 80.0        # kustdjup [m]
shelf_width = 900e3   # bredd på sluttande hylla [m]

H = shelf_bathymetry(
    grid,
    H_deep=H_deep,
    H_coast=H_coast,
    shelf_width=shelf_width,
    coast="east",
    power=1.4,
)

params = ModelParams(H=H, f0=0.0, beta=0.0, r=0.0, linear=True)

x_km = grid.x_c / 1000


### Lösningskommentar

Det viktigaste valet här är att `H` inte är ett enda tal. Det är ett fält som varierar i x-led. Därför kan modellen visa hur vågens hastighet förändras när botten blir grundare.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x_km, H[Ny // 2, :])
ax.invert_yaxis()
ax.set_xlabel("avstånd österut [km]")
ax.set_ylabel("djup H [m]")
ax.set_title("Bottenprofil: djup ocean till grund kust")
ax.grid(True)
plt.show()


## 3. Vågens teoretiska hastighet

För långa grunda-vatten-vågor är en enkel uppskattning av hastigheten

$$c = \sqrt{gH}.$$

Det betyder att vågen är snabbare där vattnet är djupt.


In [ ]:
c = wave_speed(grid, params)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x_km, c[Ny // 2, :])
ax.set_xlabel("avstånd österut [km]")
ax.set_ylabel("våghastighet sqrt(gH) [m/s]")
ax.set_title("Långa vågor går snabbast på djupt vatten")
ax.grid(True)
plt.show()

print(f"Hastighet i djup ocean:  {np.sqrt(params.g * H_deep):6.1f} m/s")
print(f"Hastighet nära kusten:   {np.sqrt(params.g * H_coast):6.1f} m/s")
print(f"Djupvattensvågen är ungefär {np.sqrt(H_deep / H_coast):.1f} gånger snabbare.")


### Svar på frågan: Varför är tsunamin snabbast ute på djupt vatten?

Eftersom hastigheten ungefär är `sqrt(g H)`. När `H` är stort blir `sqrt(g H)` också stort. I detta exempel är djupet 4000 m ute i oceanen och bara 80 m nära kusten, så vågen går mycket snabbare på djupt vatten.


## 4. Starta en tsunami-liknande våg

Vi använder en bred upphöjning av vattenytan som en enkel representation av ett jordskalv som har lyft havsytan. Den är inte tänkt att vara en realistisk jordbävningsmodell.


In [ ]:
def tsunami_initial_condition(grid, params):
    return setup_initial_state(
        grid,
        params,
        mode="gaussian_bump",
        amp=0.20,          # startamplitud [m]
        R=120e3,           # vågens bredd [m]
        x0=450e3,          # startar i djup ocean
        y0=0.5 * grid.Ly,
    )

eta0, u0, v0 = tsunami_initial_condition(grid, params)

plt.figure(figsize=(8, 3))
plt.pcolormesh(grid.x_c / 1000, grid.y_c / 1000, eta0, shading="auto")
plt.colorbar(label="eta [m]")
plt.xlabel("x [km]")
plt.ylabel("y [km]")
plt.title("Startläge: förenklad havsytelyftning")
plt.show()


## 5. Lägg till ett svamplager

Svamplagret ligger i väster, alltså på vänster sida. Det dämpar vågen som går åt fel håll, så att den inte studsar tillbaka och stör experimentet.


In [ ]:
sponge_width = 300e3
sponge = make_sponge_hook(width=sponge_width, tau=900.0, sides=("west",), power=2.0)
mask = sponge_mask_eta(grid, width=sponge_width, sides=("west",), power=2.0)

plt.figure(figsize=(8, 2.5))
plt.pcolormesh(grid.x_c / 1000, grid.y_c / 1000, mask, shading="auto", vmin=0, vmax=1)
plt.colorbar(label="svampstyrka")
plt.xlabel("x [km]")
plt.ylabel("y [km]")
plt.title("Svamplager vid västra randen")
plt.show()


### Svar på frågan: Vad händer om svamplagret tas bort?

Utan svamplagret reflekteras mer energi från den västra randen. Då kan vågor studsa tillbaka in i bassängen och göra det svårare att tolka vad som beror på uppgrundning nära kusten.


## 6. Kör modellen

Tidssteget väljs med CFL-villkoret. Eftersom vågen är snabbast på djupt vatten måste tidssteget passa den djupa delen av bassängen.


In [ ]:
dt = compute_dt_cfl(grid, params, cfl=0.45)
tmax = 4.0 * 3600.0
save_every = max(1, int((5 * 60) / dt))

print(f"dt = {dt:.1f} s")
print(f"save_every = {save_every} tidssteg")

out = run_model(
    tmax,
    dt,
    grid,
    params,
    zero_forcing,
    tsunami_initial_condition,
    save_every=save_every,
    hooks=[sponge],
    show_progress=True,
)


## 7. Animation

Kör cellen nedan för att skapa en animation i `animations/`. Den mappen bör inte spåras av git.


In [ ]:
anim = animate_eta(
    out,
    grid,
    interval=120,
    title="Tsunami med uppgrundning och svamplager",
    contours=True,
    show_colorbar=True,
    remove_mean=False,
)

# Spara som gif. Kommentera bort raden om du bara vill visa animationen i notebooken.
anim.save(str(ANIMATION_DIR / "02_tsunami_shoaling_sv_solution.gif"), fps=10)
anim


## 8. Mät våghöjd längs x-led

Vi mäter den största absoluta havsytan som förekommer vid varje x-position under hela körningen.


In [ ]:
eta_stack = np.stack(out["eta"], axis=0)
max_abs_eta_x = np.max(np.abs(eta_stack), axis=(0, 1))

fig, ax1 = plt.subplots(figsize=(8, 3.5))
ax1.plot(x_km, max_abs_eta_x, label="största |eta|")
ax1.set_xlabel("avstånd österut [km]")
ax1.set_ylabel("max |eta| [m]")
ax1.grid(True)
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(x_km, H[Ny // 2, :], linestyle="--", label="djup")
ax2.invert_yaxis()
ax2.set_ylabel("djup H [m]")
ax2.legend(loc="upper right")

plt.title("Våghöjden ökar när havet blir grundare")
plt.show()


### Svar på frågan: Var blir vågen högst?

I en typisk körning blir den största våghöjden nära den grundare delen mot öster. Det är där vågen bromsas upp av det mindre djupet. Eftersom vattenmassan bakom vågen fortfarande kommer framåt kan vattenytan höjas mer där än ute på djupt vatten.

Resultatet påverkas av upplösning, kustdjup, hyllans bredd och hur länge modellen körs, men den viktiga poängen är att våghöjden brukar öka när vågen kommer in över grundare vatten.


## 9. Utmaning: ändra kustdjup

Labben föreslog tre kustdjup:

- `H_coast = 200.0`
- `H_coast = 80.0`
- `H_coast = 30.0`

Koden nedan gör en enkel jämförelse. För att hålla notebooken snabb är standardinställningen att bara visa de teoretiska hastigheterna. Sätt `RUN_EXTRA_CASES = True` om du vill köra alla tre modellerna.


In [ ]:
coastal_depths = [200.0, 80.0, 30.0]

for Hc in coastal_depths:
    c_coast = np.sqrt(params.g * Hc)
    shoaling_ratio = (H_deep / Hc) ** 0.25
    print(
        f"H_coast = {Hc:5.1f} m: "
        f"c_coast = {c_coast:5.1f} m/s, "
        f"enkel shoaling-faktor ~ {shoaling_ratio:4.2f}"
    )


### Lösning på utmaningen

Det grundaste fallet, `H_coast = 30 m`, bör ge den tydligaste uppgrundningen och ofta den största vågen nära kusten.

Förklaringen är:

- mindre **djup** ger lägre våghastighet,
- vågen bromsas när **hastigheten** minskar,
- när vågenergin trängs ihop i grundare vatten kan amplituden öka: detta är **uppgrundning**.

Den enkla skalningen `amplitud ~ H^(-1/4)` säger också att lägre djup bör ge större amplitud, även om den exakta siffran i modellen beror på hela experimentet.


In [ ]:
RUN_EXTRA_CASES = False

def run_case(H_coast_case):
    H_case = shelf_bathymetry(
        grid,
        H_deep=H_deep,
        H_coast=H_coast_case,
        shelf_width=shelf_width,
        coast="east",
        power=1.4,
    )
    params_case = ModelParams(H=H_case, f0=0.0, beta=0.0, r=0.0, linear=True)
    sponge_case = make_sponge_hook(width=sponge_width, tau=900.0, sides=("west",), power=2.0)
    dt_case = compute_dt_cfl(grid, params_case, cfl=0.45)
    save_every_case = max(1, int((5 * 60) / dt_case))

    out_case = run_model(
        tmax,
        dt_case,
        grid,
        params_case,
        zero_forcing,
        tsunami_initial_condition,
        save_every=save_every_case,
        hooks=[sponge_case],
        show_progress=False,
    )
    eta_case = np.stack(out_case["eta"], axis=0)
    return np.max(np.abs(eta_case), axis=(0, 1))

if RUN_EXTRA_CASES:
    plt.figure(figsize=(8, 3.5))
    for Hc in coastal_depths:
        max_eta = run_case(Hc)
        plt.plot(x_km, max_eta, label=f"H_coast = {Hc:.0f} m")
    plt.xlabel("avstånd österut [km]")
    plt.ylabel("max |eta| [m]")
    plt.title("Jämförelse av olika kustdjup")
    plt.grid(True)
    plt.legend()
    plt.show()


## 10. Sammanfattande facit

**Vilken parameter ändrade du?**  
Ett bra exempel är kustdjupet `H_coast`.

**Vad borde hända?**  
När kustdjupet minskar borde vågen gå långsammare nära kusten och uppgrundningen bli tydligare.

**Vad hände i modellen?**  
Vågen rör sig snabbast över djupt vatten. När den kommer in på grundare vatten minskar hastigheten och den största vattenståndsändringen blir ofta större nära kusten.

**Vilken figur visar detta tydligast?**  
Figuren med `max |eta|` tillsammans med bottenprofilen är den bästa diagnosen. Animationen är bäst för en intuitiv bild.

**Varför visar modellen inte hur långt vattnet rinner upp på land?**  
Modellen har bara vatten över en fast botten. Den har ingen torr mark, ingen vågbrytning och ingen våt/torr-gräns. Därför kan den visa uppgrundning, men inte verklig översvämning eller run-up på land.
